# PRE PROCESSING

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path("..").resolve()

# Read and preview all NHAMCS .dta files in the local data folder
data_dir = PROJECT_ROOT / "data"
dta_files = sorted(data_dir.glob("*.dta"))

if not dta_files:
    raise FileNotFoundError(f"No .dta files found in {data_dir.resolve()}")

print(f"Found {len(dta_files)} .dta files:")
for f in dta_files:
    print(f"- {f.name}")

Found 5 .dta files:
- ED2018-stata.dta
- ED2019-stata.dta
- ed2020-stata.dta
- ed2021-stata.dta
- ed2022-stata.dta


In [2]:
required_columns = [
    #time and date of arrival 
    'VMONTH', 'VDAYR','ARRTIME','ARREMS', 'VITALSD',
    # demographics 
    'AGE', 'RESIDNCE', 'SEX', 'RACERETH', 'PAYTYPER', 'NOPAY', 'REGION', 'RACER', 'RACEUN' , 'ETHIM', 'ETHUN',
    # vitals
    'TEMPF', 'PULSE', 'RESPR', 'BPSYS', 'BPDIAS', 'POPCT','PAINSCALE',
    # clinical information
    'SEEN72', 'RFV1', 'RFV2', 'RFV3', 'RFV4', 'RFV5', 'EPISODE',
    # injuries  
    'INJPOISAD', 'CAUSE1', 'CAUSE2', 'CAUSE3',
    # chronic conditions
    'ALZHD', 'ASTHMA', 'CANCER', 'CEBVD', 'CKD', 'COPD', 'CHF', 'CAD', 'DEPRN', 'DIABTYP1',
    'DIABTYP2', 'DIABTYP0', 'ESRD', 'HPE', 'EDHIV', 'HYPLIPID', 'HTN', 'OBESITY', 'OSA', 'OSTPRSIS', 
    # substance abuse 
    'SUBSTAB',
    # target variables
    'IVFLUIDS','IMMEDR', 'WAITTIME'
    ]

In [ ]:
data_dir = PROJECT_ROOT / "data"

year_configs = {
    2018: data_dir / "ED2018-stata.dta",
    2019: data_dir / "ED2019-stata.dta",
    2020: data_dir / "ed2020-stata.dta",
    2021: data_dir / "ed2021-stata.dta",
    2022: data_dir / "ed2022-stata.dta",
}

dfs = []
for year, path in year_configs.items():
    # 1. Load data as numbers
    reader = pd.read_stata(path, convert_categoricals=False, iterator=True)
    df = reader.read()

    df = df.loc[:, required_columns]
    # 2. Extract the labels dictionary
    # This stays as a dictionary of dictionaries: { 'column_name': {value: 'label'} }
    labels = reader.value_labels()
    # Create a new, clean version of the labels dictionary
    clean_labels = {}

    for format_name, mapping in labels.items():
        clean_labels[format_name] = {
            # Convert np.int32, np.int64, or floats to standard int
            int(k): str(v) for k, v in mapping.items() 
            if pd.notna(k) and str(k).replace('.0', '').lstrip('-').isdigit()
        }

    # Update your master labels
    labels = clean_labels
    dfs.append((df, labels))
    

In [4]:
dfs[0][1]["RESIDF"]

{-9: 'Blank',
 -8: 'Unknown',
 1: 'Private residence',
 2: 'Nursing home',
 3: 'Homeless/homeless shelter',
 4: 'Other'}

In [5]:
import pandas as pd
import numpy as np

columns_to_fix = [
    'RESIDNCE', 'TEMPF', 'SEX', 'PAYTYPER', 
    'RFV1', 'RFV2', 'RFV3', 'RFV4', 'RFV5', 
    'EPISODE', 'ARREMS', 'REGION'
]

new_frames = []

for df, labels_dict in dfs:
    # 1. Sanitize the labels: Force keys to standard Python integers
    # This is crucial so that -9.0 (float) or np.int32 matches the dictionary
    sanitized_labels = {}
    for fmt_name, mapping in labels_dict.items():
        sanitized_labels[fmt_name] = {
            int(k): str(v) for k, v in mapping.items() 
            if str(k).replace('.0', '').lstrip('-').isdigit()
        }

    for col in columns_to_fix:
        if col not in df.columns:
            continue

        # Determine the correct dictionary key from the CDC naming conventions
        label_key = col if col in sanitized_labels else col + 'F'
        if col.startswith('RFV'):
            label_key = 'RFVF'
        if col == 'RESIDNCE':
            label_key = 'RESIDF'

        if label_key in sanitized_labels:
            # 2. Convert to numeric (handling floats/NaNs), fill with -9 (Blank)
            # We must convert to int before mapping to match our sanitized keys
            clean_numeric = pd.to_numeric(df[col], errors='coerce').fillna(-9).astype(int)
            
            # 3. OVERWRITE the original column with the mapped string label
            # We use .astype(str) to ensure the column is treated as a string object
            df[col] = clean_numeric.map(sanitized_labels[label_key]).fillna("Unknown").astype(str)

    new_frames.append(df)

In [6]:
# Define the detailed mapping for Race (from RACEUNF)
race_detail_map = {
    1: 'White',
    2: 'Black/African American',
    3: 'Asian',
    4: 'Native Hawaiian/PI',
    5: 'American Indian/AK Native',
    6: 'Multiple'
}

# Define Hispanic Status (from ETHUNF/ETHIMF)
hispanic_map = {1: 'Hispanic', 2: 'Not Hispanic'}

def create_accurate_race(row):
    # 1. Check Hispanic status first
    # 1 is Hispanic in NHAMCS
    if row['ETHUN'] == 1 or row['ETHIM'] == 1:
        return 'Hispanic'
    
    # 2. If not Hispanic, use the specific race category
    race_val = row['RACEUN']
    if race_val in race_detail_map:
        return race_detail_map[race_val]
    
    return 'Unknown/Other'

for frame in new_frames:
    frame['race'] = frame.apply(create_accurate_race, axis=1)
    frame['TEMPF'] = pd.to_numeric(frame['TEMPF'], errors='coerce')

In [7]:
pay_mapping = {
    'Private insurance': 'Private',
    'Medicare': 'Medicare',
    'Medicaid or CHIP or other state-based program': 'Medicaid/Public',
    'Self-pay': 'Self_Pay',
    'Worker\'s compensation': 'Workers_Comp',
    'No charge/Charity': 'Charity',
    'Other': 'Other',
    'Unknown': 'Unknown/Blank',
    'All sources of payment are blank': 'Unknown_Blank'
}

for frame in new_frames:
    frame['PAYTYPER'] = frame['PAYTYPER'].map(pay_mapping)

In [8]:
import re

def parse_sas_formats(file_path):
    formats = {}
    current_format = None
    
    # regex for VALUE $CAUSEF
    header_re = re.compile(r"VALUE\s+\$?(?P<name>\w+)")
    
    # FIXED regex: Look for 'Key'='Full Sentence Description'
    # It looks for characters inside the quotes specifically
    mapping_re = re.compile(r"'(?P<key>[^']*)'\s*=\s*'(?P<val>[^']*)'")

    with open(file_path, 'r') as f:
        for line in f:
            line = line.strip()
            
            header_match = header_re.search(line)
            if header_match:
                current_format = header_match.group('name')
                formats[current_format] = {}
                continue
            
            if current_format:
                mapping_match = mapping_re.search(line)
                if mapping_match:
                    key = mapping_match.group('key')
                    val = mapping_match.group('val')
                    formats[current_format][key] = val
            
            if ';' in line:
                current_format = None
 
    return formats

cause_labels = []
for year in range(18, 23):
    cause_labels.append(parse_sas_formats(f'format/ed{year}for.txt')['CAUSEF'])
    

In [9]:
for i in range(len(new_frames)):
    df = new_frames[i]
    labels = cause_labels[i]
    labels['-9'] = 'Unknown/Blank'
    for col in ['CAUSE1', 'CAUSE2', 'CAUSE3']:
        df[col] = df[col].astype(str)
        df[col] = df[col].map(labels)
    new_frames[i] = df


In [10]:
new_frames[0].CAUSE1.nunique()

289

In [11]:
for i in range(len(new_frames)):
    new_frames[i]['year'] = 2018 + i

In [12]:
result = pd.concat(new_frames, ignore_index=True)

In [13]:
rename_mapping = {
    # Logistics
    'VMONTH': 'visit_month', 'VDAYR': 'day_of_week', 'ARRTIME': 'arrival_time', 'ARREMS': 'ems_arrival',
    # Demographics
    'AGE': 'age', 'SEX': 'sex', 'PAYTYPER': 'insurance', 'RESIDNCE': 'residence',
    # Vitals
    # temperature is in celcius
    'TEMPF': 'temp', 'PULSE': 'heart_rate', 'RESPR': 'resp_rate', 'BPSYS': 'sys_bp', 
    'BPDIAS': 'dias_bp', 'POPCT': 'spo2', 'PAINSCALE': 'pain_score', 'VITALSD': 'vitals_during_visit',
    # Visit Details
    'RFV1': 'complaint_1', 'RFV2': 'complaint_2', 'RFV3': 'complaint_3', 'RFV4': 'complaint_4', 'RFV5': 'complaint_5',
    'SEEN72': 'seen_last_72h', 'EPISODE': 'episode', 'INJPOISAD': 'is_injury_poison',
    'CAUSE1': 'injury_cause_1', 'CAUSE2': 'injury_cause_2', 'CAUSE3': 'injury_cause_3',
    # Comorbidities
    'ALZHD': 'hist_alzheimers', 'ASTHMA': 'hist_asthma', 'CANCER': 'hist_cancer', 'CEBVD': 'hist_stroke',
    'CKD': 'hist_ckd', 'COPD': 'hist_copd', 'CHF': 'hist_chf', 'CAD': 'hist_cad', 'DEPRN': 'hist_depression',
    'DIABTYP1': 'hist_diabetes_t1', 'DIABTYP2': 'hist_diabetes_t2', 'DIABTYP0': 'hist_diabetes_unspec',
    'ESRD': 'hist_esrd', 'HPE': 'hist_pe', 'EDHIV': 'hist_hiv', 'HYPLIPID': 'hist_high_cholesterol',
    'HTN': 'hist_hypertension', 'OBESITY': 'hist_obesity', 'OSA': 'hist_sleep_apnea', 
    'OSTPRSIS': 'hist_osteoporosis', 'SUBSTAB': 'hist_substance_abuse',
    # Interventions & Target
    'IVFLUIDS': 'intervention_iv_fluids',
    'IMMEDR': 'target_triage_acuity',
    'WAITTIME': 'wait_time_minutes',
    'REGION': 'region',
    'NOPAY': 'no_payment',

}

# Apply the renaming
result.rename(columns=rename_mapping, inplace=True)

# Let's see the clean columns!
print(result.columns)

Index(['visit_month', 'day_of_week', 'arrival_time', 'ems_arrival',
       'vitals_during_visit', 'age', 'residence', 'sex', 'RACERETH',
       'insurance', 'no_payment', 'region', 'RACER', 'RACEUN', 'ETHIM',
       'ETHUN', 'temp', 'heart_rate', 'resp_rate', 'sys_bp', 'dias_bp', 'spo2',
       'pain_score', 'seen_last_72h', 'complaint_1', 'complaint_2',
       'complaint_3', 'complaint_4', 'complaint_5', 'episode',
       'is_injury_poison', 'injury_cause_1', 'injury_cause_2',
       'injury_cause_3', 'hist_alzheimers', 'hist_asthma', 'hist_cancer',
       'hist_stroke', 'hist_ckd', 'hist_copd', 'hist_chf', 'hist_cad',
       'hist_depression', 'hist_diabetes_t1', 'hist_diabetes_t2',
       'hist_diabetes_unspec', 'hist_esrd', 'hist_pe', 'hist_hiv',
       'hist_high_cholesterol', 'hist_hypertension', 'hist_obesity',
       'hist_sleep_apnea', 'hist_osteoporosis', 'hist_substance_abuse',
       'intervention_iv_fluids', 'target_triage_acuity', 'wait_time_minutes',
       'race', 'year

In [14]:
result.drop(columns=['RACERETH', 'RACER', 'RACEUN', 'ETHIM', 'ETHUN'], inplace=True)

In [15]:
import pandas as pd

print("Starting Integrated NLP Text Preprocessing...")

# 1. Column definitions
complaint_cols = ['complaint_1', 'complaint_2', 'complaint_3', 'complaint_4', 'complaint_5']
injury_cols = ['injury_cause_1', 'injury_cause_2', 'injury_cause_3']

def clean_and_merge_integrated(row, columns):
    text_parts = []
    
    junk_strings = {
        '-9', '-8', 'blank', 'unknown', 
        'unknown/blank', 'none', '', 'nan', 'not otherwise specified'
    }
    
    for col in columns:
        if col in row.index:
            val = row[col]
            if pd.notna(val):
                val_str = str(val).strip().lower()
                
                # Remove NOS
                if val_str.endswith(', nos'):
                    val_str = val_str[:-5]
                elif val_str.endswith(' nos'):
                    val_str = val_str[:-4]
                
                # Keep only meaningful text (must contain letters)
                if val_str not in junk_strings and any(c.isalpha() for c in val_str):
                    text_parts.append(val_str)
    
    return ". ".join(text_parts) if text_parts else "Blank/Unknown"

# 4. Apply to the dataframe
result['chief_complaint_text'] = result.apply(lambda row: clean_and_merge_integrated(row, complaint_cols), axis=1)
result['injury_cause_text'] = result.apply(lambda row: clean_and_merge_integrated(row, injury_cols), axis=1)

# 5. Drop the raw columns to save memory
cols_to_drop = [c for c in complaint_cols + injury_cols if c in result.columns]
result.drop(columns=cols_to_drop, inplace=True)

print(f"Merge complete. Final shape: {result.shape}")
print(result[['chief_complaint_text', 'injury_cause_text']].head(10))

Starting Integrated NLP Text Preprocessing...
Merge complete. Final shape: (86864, 49)
                                chief_complaint_text  \
0                             fever. throat soreness   
1  injury, other and unspecified, of foo.... foot...   
2                                       fever. cough   
3  abdominal pain, cramps, spasms. chest pain. po...   
4           injury, other and unspecified, of foo...   
5                                              fever   
6                            cough. nasal congestion   
7                                              cough   
8           injury, other and unspecified of head...   
9                                          skin rash   

                                   injury_cause_text  
0                                      Blank/Unknown  
1  other foreign body or object entering through ...  
2                                      Blank/Unknown  
3                                      Blank/Unknown  
4  overexertion from 

In [16]:
result.columns

Index(['visit_month', 'day_of_week', 'arrival_time', 'ems_arrival',
       'vitals_during_visit', 'age', 'residence', 'sex', 'insurance',
       'no_payment', 'region', 'temp', 'heart_rate', 'resp_rate', 'sys_bp',
       'dias_bp', 'spo2', 'pain_score', 'seen_last_72h', 'episode',
       'is_injury_poison', 'hist_alzheimers', 'hist_asthma', 'hist_cancer',
       'hist_stroke', 'hist_ckd', 'hist_copd', 'hist_chf', 'hist_cad',
       'hist_depression', 'hist_diabetes_t1', 'hist_diabetes_t2',
       'hist_diabetes_unspec', 'hist_esrd', 'hist_pe', 'hist_hiv',
       'hist_high_cholesterol', 'hist_hypertension', 'hist_obesity',
       'hist_sleep_apnea', 'hist_osteoporosis', 'hist_substance_abuse',
       'intervention_iv_fluids', 'target_triage_acuity', 'wait_time_minutes',
       'race', 'year', 'chief_complaint_text', 'injury_cause_text'],
      dtype='str')

In [17]:
import pandas as pd
import numpy as np

def nuke_special_codes(df):
    print("=== BEFORE ===")
    
    # 1. Define the exact garbage values we want to target
    # Catching ints, floats, and common string formats
    target_values = [
        -9, -8, -7, -6, -5, -1, 
        -9.0, -8.0, -7.0, -6.0, -5.0, -1.0,
        '-9', '-8', '-7', '-6', '-5', '-1',
        '-9.0', '-8.0', '-7.0', '-6.0', '-5.0', '-1.0'
    ]
    
    # 2. Count occurrences BEFORE replacing
    before_mask = df.isin(target_values)
    total_before = before_mask.sum().sum()
    print(f"Total special codes found: {total_before}")
    
    if total_before > 0:
        print("\nTop 5 columns containing these codes:")
        print(before_mask.sum().sort_values(ascending=False).head(5))
        
    print("\nExecuting replacement...")
    
    # 3. Replace the target values with NaN
    cleaned_df = df.replace(target_values, np.nan)
    
    print("\n=== AFTER ===")
    # 4. Count total NaNs AFTER replacing
    total_nans_after = cleaned_df.isna().sum().sum()
    print(f"Total NaN values in dataframe: {total_nans_after}")
    
    # Show the difference to confirm it worked
    print(f"Net change in NaNs: +{total_nans_after - df.isna().sum().sum()}")
    
    return cleaned_df

# Run the function on your dataframe
ml_df = nuke_special_codes(result)

=== BEFORE ===
Total special codes found: 123490

Top 5 columns containing these codes:
pain_score              34758
target_triage_acuity    22264
wait_time_minutes       13147
dias_bp                 10051
sys_bp                   9938
dtype: int64

Executing replacement...

=== AFTER ===
Total NaN values in dataframe: 129449
Net change in NaNs: +123490


In [18]:
print(f"Rows before dropping target NaNs: {len(ml_df)}")

# Drop rows where the target variable is missing
ml_df = ml_df.dropna(subset=['target_triage_acuity'])

# Ensure the target is formatted as a clean integer (1, 2, 3, 4, 5)
# Keep ONLY the valid clinical triage scores
ml_df = ml_df[ml_df['target_triage_acuity'].isin([1, 2, 3, 4, 5])]

print(f"Rows after dropping target NaNs: {len(ml_df)}")
print("\nFinal Target Distribution:")
print(ml_df['target_triage_acuity'].value_counts().sort_index())

Rows before dropping target NaNs: 86864
Rows after dropping target NaNs: 58124

Final Target Distribution:
target_triage_acuity
1.0      846
2.0     8597
3.0    29568
4.0    16715
5.0     2398
Name: count, dtype: int64


In [ ]:
output_path = PROJECT_ROOT / "working_data" / "nhamcs_data_2018_22.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
ml_df.to_csv(output_path, index=False)
print(f"Saved to {output_path}")

In [22]:
dfs[0][1]["INJPOISADF"]

{1: 'Yes, injury/trauma',
 2: 'Yes, overdose/poisoning',
 3: 'Yes, adverse effect of medical/surgical treatment',
 4: 'No, visit is not related to injury/trauma, overdose/poisoning, or adverse effe',
 5: 'Questionable injury status',
 -8: 'Unknown',
 -9: 'Blank'}